# Customer Segmentation and Predictive Analytics
**Name:** Shravni Umrani  
**Roll number:** 23102C0058  
**Course:** R Programming | Uploaded problem statement 7 (PDF heading: Lab Problem Statement 6)

## Aim and problem statement
Use UCI Online Retail transactions to build customer features, discover purchasing segments and classify membership in a high-value segment. Compare K-Means with hierarchical clustering; visualize clusters with PCA and Plotly; compare Random Forest and SVM using classification metrics and recommend marketing actions.

## How to run
Open this file in Google Colab and select **Runtime → Run all**. The first run downloads the real dataset; if download fails, upload `Online Retail.xlsx` when prompted. Results and graphs are generated by execution, rather than supplied as invented outputs.

The PDF explicitly requests Python in Colab. Its later predictive-maintenance passages provide no separate dataset or complete problem statement; this notebook implements the fully specified customer analytics experiment.

## Dataset
UCI Online Retail: 541,909 transaction rows from a UK-based online retailer, December 2010–December 2011. Fields: InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID and Country. Source: https://archive.ics.uci.edu/dataset/352/online+retail


In [ ]:
import sys, subprocess, importlib.util
for module, package in [("openpyxl", "openpyxl"), ("plotly", "plotly")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (silhouette_score, adjusted_rand_score, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance
from scipy.cluster.hierarchy import linkage, dendrogram
import plotly.express as px
from pathlib import Path
SEED = 42
sns.set_theme(style="whitegrid")


## 1. Load and inspect the dataset

In [ ]:
import urllib.request, zipfile
xlsx = Path("Online Retail.xlsx")
if not xlsx.exists():
    try:
        urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/352/online+retail.zip", "retail.zip")
        with zipfile.ZipFile("retail.zip") as z:
            member = next(x for x in z.namelist() if x.lower().endswith(".xlsx"))
            xlsx.write_bytes(z.read(member))
    except Exception as exc:
        print("Automatic download failed:", exc)
        from google.colab import files
        uploaded = files.upload()
        candidate = next((x for x in uploaded if x.lower().endswith(".xlsx")), None)
        if candidate is None:
            raise ValueError("Please upload the UCI Online Retail Excel file.")
        xlsx = Path(candidate)
raw = pd.read_excel(xlsx)
print("Dataset shape:", raw.shape)
display(raw.head())
display(raw.isna().sum().to_frame("Missing values"))


## 2. Preprocessing and feature engineering
Remove exact duplicate rows, missing customer IDs, invalid dates, cancelled invoices (prefix C), nonpositive quantities and nonpositive prices. Missing product descriptions are retained because they are not needed for customer features. This analysis describes completed positive purchases; refunds are excluded rather than subtracted.

Recency is days since the last purchase; frequency is the number of distinct invoices; monetary value is total spend. We also calculate total quantity, average invoice value, active days and monthly purchase rate. Purchase rate uses at least 30 days of observation to avoid extreme values for very new customers.

In [ ]:
df = raw.drop_duplicates().copy()
audit = {"Original rows": len(raw), "After duplicate removal": len(df)}
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df = df.dropna(subset=["CustomerID", "InvoiceNo", "InvoiceDate", "Quantity", "UnitPrice"])
audit["After required-value filtering"] = len(df)
df = df[~df["InvoiceNo"].astype(str).str.upper().str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()
audit["Completed positive purchase rows"] = len(df)
df["CustomerID"] = df["CustomerID"].astype(int)
df["LineTotal"] = df["Quantity"] * df["UnitPrice"]
snapshot = df["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)
customers = df.groupby("CustomerID").agg(
    LastPurchase=("InvoiceDate", "max"), FirstPurchase=("InvoiceDate", "min"),
    Frequency=("InvoiceNo", "nunique"), Monetary=("LineTotal", "sum"),
    TotalQuantity=("Quantity", "sum"))
customers["Recency"] = (snapshot - customers["LastPurchase"]).dt.days
customers["ActiveDays"] = (snapshot - customers["FirstPurchase"]).dt.days.clip(lower=1)
customers["AverageTransactionValue"] = customers["Monetary"] / customers["Frequency"]
customers["PurchaseFrequencyMonthly"] = customers["Frequency"] / (customers["ActiveDays"].clip(lower=30) / 30)
features = ["Recency", "Frequency", "Monetary", "AverageTransactionValue", "TotalQuantity", "PurchaseFrequencyMonthly"]
X = customers[features].copy()
assert np.isfinite(X.to_numpy()).all() and (X >= 0).all().all()
display(pd.Series(audit, name="Rows").to_frame())
display(X.head())
display(X.describe().round(2))


## 3. Split customers before learning preprocessing
Reserve 25% of customers as a test set. Fit outlier limits, scaling, selection of cluster count and K-Means on training customers only. Test customers receive labels from the trained K-Means model. This prevents test data from influencing the segmentation rules.

Winsorize at training-set 1st/99th percentiles, apply log1p to reduce skew, then standardize. Original values are retained for business profiles. Correlated derived features are included to meet the assignment; they may give spending/activity more weight.

In [ ]:
train_ids, test_ids = train_test_split(X.index.to_numpy(), test_size=0.25, random_state=SEED)
X_train, X_test = X.loc[train_ids], X.loc[test_ids]
low, high = X_train.quantile(0.01), X_train.quantile(0.99)
def log_features(frame):
    return np.log1p(frame.clip(lower=low, upper=high, axis=1))
scaler = StandardScaler()
Z_train = scaler.fit_transform(log_features(X_train))
Z_test = scaler.transform(log_features(X_test))
Z_all = scaler.transform(log_features(X))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(X_train["Monetary"], bins=50, ax=axes[0]); axes[0].set_title("Original customer spend")
sns.histplot(log_features(X_train)["Monetary"], bins=50, ax=axes[1]); axes[1].set_title("Capped, log-transformed spend")
plt.tight_layout(); plt.show()


## 4. Elbow Method, silhouette scores and K-Means
The elbow plot checks diminishing improvements in inertia. The automatic choice uses the highest sampled silhouette score among k=2…8; the elbow and business profiles should also inform practical selection.

In [ ]:
rows, models = [], {}
for k in range(2, 9):
    model = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(Z_train)
    score = silhouette_score(Z_train, model.labels_, sample_size=min(1500, len(Z_train)), random_state=SEED)
    rows.append({"k": k, "Inertia": model.inertia_, "Silhouette": score})
    models[k] = model
selection = pd.DataFrame(rows)
best_k = int(selection.loc[selection["Silhouette"].idxmax(), "k"])
km = models[best_k]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(selection.k, selection.Inertia, "o-"); axes[0].set(xlabel="Number of clusters", ylabel="Inertia", title="Elbow Method")
axes[1].plot(selection.k, selection.Silhouette, "o-"); axes[1].set(xlabel="Number of clusters", ylabel="Silhouette score", title="Cluster separation")
plt.tight_layout(); plt.show()
display(selection.round(4)); print("Selected k:", best_k)
customers["Cluster"] = km.predict(Z_all)


## 5. Hierarchical clustering and dendrogram
Use the same reproducible sample (up to 1,000 training customers) for both methods so their silhouette comparison is fair. Ward linkage merges groups that minimally increase within-cluster variance. Sampling keeps the dendrogram readable and limits memory usage.

In [ ]:
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(Z_train), size=min(1000, len(Z_train)), replace=False)
Z_sample = Z_train[sample_idx]
ward = linkage(Z_sample, method="ward")
plt.figure(figsize=(13, 5))
dendrogram(ward, truncate_mode="lastp", p=30, show_leaf_counts=True)
plt.title("Ward dendrogram (truncated to 30 groups)"); plt.xlabel("Merged customer groups"); plt.ylabel("Ward distance")
plt.tight_layout(); plt.show()
h_labels = AgglomerativeClustering(n_clusters=best_k, linkage="ward").fit_predict(Z_sample)
k_labels = km.predict(Z_sample)
comparison = pd.DataFrame({"Method": ["K-Means", "Hierarchical (Ward)"],
    "Silhouette": [silhouette_score(Z_sample, k_labels), silhouette_score(Z_sample, h_labels)]})
display(comparison.round(4))
print("Adjusted Rand Index (agreement between methods):", round(adjusted_rand_score(k_labels, h_labels), 4))
print("Higher silhouette indicates clearer separation; cluster IDs are arbitrary between methods.")


## 6. PCA in 2D and interactive Plotly in 3D
PCA is fitted on training customers and used for visualization. Clustering remains in the full feature space.

In [ ]:
pca = PCA(n_components=3, random_state=SEED).fit(Z_train)
projection = pca.transform(Z_all)
vis = pd.DataFrame(projection, columns=["PC1", "PC2", "PC3"], index=X.index)
vis["Cluster"] = customers["Cluster"].astype(str)
vis["Monetary"] = customers["Monetary"]
vis["CustomerID"] = vis.index.astype(str)
print("Explained variance:", np.round(pca.explained_variance_ratio_, 4))
print("Variance retained by 2D projection:", round(pca.explained_variance_ratio_[:2].sum(), 4))
plt.figure(figsize=(10, 6))
sns.scatterplot(data=vis, x="PC1", y="PC2", hue="Cluster", palette="tab10", s=25, alpha=0.7)
plt.title("Customer segments: 2D PCA projection"); plt.tight_layout(); plt.show()
fig3d = px.scatter_3d(vis, x="PC1", y="PC2", z="PC3", color="Cluster",
    hover_data=["CustomerID", "Monetary"], title="Interactive customer segments")
fig3d.update_traces(marker_size=3); fig3d.show()
fig3d.write_html("customer_clusters_3d.html")


## 7. Cluster profiles and high-value target
Select the high-value segment using the highest median monetary value among **training** clusters. Frequency and recency provide context. These labels mean membership in a present-day spending segment, not future purchases or lifetime value.

In [ ]:
train_profiles = customers.loc[train_ids].groupby("Cluster")[features].median()
high_value_cluster = int(train_profiles["Monetary"].idxmax())
profiles = customers.groupby("Cluster")[features].median().round(2)
profiles.insert(0, "CustomerCount", customers.groupby("Cluster").size())
profiles["RevenueSharePct"] = (customers.groupby("Cluster")["Monetary"].sum() / customers["Monetary"].sum() * 100).round(2)
display(profiles)
print("High-value cluster selected using training customers:", high_value_cluster)
profile_z = (profiles[features] - X.median()) / X.std()
plt.figure(figsize=(11, 5)); sns.heatmap(profile_z, annot=True, fmt=".2f", cmap="RdYlBu_r", center=0)
plt.title("Relative cluster profiles (higher recency means less recent)"); plt.tight_layout(); plt.show()
y_train = (customers.loc[train_ids, "Cluster"] == high_value_cluster).astype(int)
y_test = (customers.loc[test_ids, "Cluster"] == high_value_cluster).astype(int)
display(pd.DataFrame({"Train": y_train.value_counts(), "Test": y_test.value_counts()}).fillna(0).astype(int))
if y_train.nunique() < 2 or y_test.nunique() < 2:
    raise ValueError("Both classes are needed in each split; choose a different random seed and rerun.")


## 8. Random Forest and SVM prediction
Both classifiers learn the same training-derived cluster membership. Fit the SVM scaler only on training data. Balanced class weights address unequal segment sizes.

**Interpretation limit:** labels are generated from these same purchasing features. Strong accuracy measures how well classifiers reproduce K-Means assignments; it is not evidence of future high-value prediction. A future prediction study needs earlier-period inputs and later-period outcomes with a temporal split.

In [ ]:
L_train, L_test = log_features(X_train), log_features(X_test)
classifiers = {
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1),
    "SVM": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, class_weight="balanced"))
}
metrics = []
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig_roc, ax_roc = plt.subplots(figsize=(7, 5))
for i, (name, model) in enumerate(classifiers.items()):
    model.fit(L_train, y_train)
    pred = model.predict(L_test)
    scores = model.predict_proba(L_test)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(L_test)
    metrics.append({"Model": name, "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-Score": f1_score(y_test, pred, zero_division=0), "ROC-AUC": roc_auc_score(y_test, scores)})
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["Other", "High-value"], ax=axes[i], colorbar=False)
    axes[i].set_title(name)
    RocCurveDisplay.from_predictions(y_test, scores, name=name, ax=ax_roc)
fig.tight_layout(); display(fig); plt.close(fig)
ax_roc.plot([0, 1], [0, 1], "k--", label="Random baseline")
ax_roc.set_title("ROC curves: held-out customers"); ax_roc.legend(); fig_roc.tight_layout(); display(fig_roc); plt.close(fig_roc)
results = pd.DataFrame(metrics).set_index("Model")
display(results.round(4))
print("Majority-class baseline accuracy:", round(max(y_test.mean(), 1-y_test.mean()), 4))
results.plot.bar(figsize=(10, 5), ylim=(0, 1.05), rot=0, title="Model performance comparison")
plt.tight_layout(); plt.show()


## 9. Feature importance
Show Random Forest impurity importance and held-out permutation importance. Correlated RFM features can share importance; association here does not establish causation.

In [ ]:
rf = classifiers["Random Forest"]
importance = pd.Series(rf.feature_importances_, index=features).sort_values()
importance.plot.barh(figsize=(8, 4), title="Random Forest feature importance")
plt.xlabel("Impurity-based importance"); plt.tight_layout(); plt.show()
perm = permutation_importance(rf, L_test, y_test, scoring="f1", n_repeats=5, random_state=SEED, n_jobs=-1)
perm_table = pd.DataFrame({"Feature": features, "Mean F1 decrease": perm.importances_mean,
    "Std": perm.importances_std}).sort_values("Mean F1 decrease", ascending=False)
display(perm_table.round(4))


## 10. Segment-wise interpretation and marketing recommendations
The following recommendations are generated from the actual cluster medians. Higher recency indicates a longer period since the last purchase.

In [ ]:
recommendations = []
for cluster, row in profiles.iterrows():
    if cluster == high_value_cluster:
        label, strategy = "High-value customers", "VIP service, early access, loyalty rewards and personalized bundles; avoid unnecessary blanket discounts."
    elif row["Recency"] > X_train["Recency"].quantile(0.75):
        label, strategy = "Inactive / at-risk customers", "Run a limited win-back offer and ask for feedback; test response before increasing spend."
    elif row["Frequency"] >= X_train["Frequency"].median() and row["Recency"] <= X_train["Recency"].median():
        label, strategy = "Active repeat customers", "Use loyalty points, relevant cross-selling and replenishment reminders."
    elif row["Recency"] <= X_train["Recency"].median():
        label, strategy = "Recent low-frequency customers", "Send onboarding messages and a second-purchase incentive with relevant products."
    else:
        label, strategy = "Occasional customers", "Use seasonal campaigns and affordable bundles; limit contact frequency."
    recommendations.append({"Cluster": cluster, "Profile": label, "Marketing action": strategy})
recommendation_table = pd.DataFrame(recommendations)
display(recommendation_table)
print("Best silhouette on the common sample:", comparison.loc[comparison.Silhouette.idxmax(), "Method"])
print("Classifier with highest held-out F1:", results["F1-Score"].idxmax())
print("Check recall for missed high-value customers and precision for wasted campaign spending.")
profiles.to_csv("customer_cluster_profiles.csv")
results.to_csv("model_performance.csv")
recommendation_table.to_csv("marketing_recommendations.csv", index=False)


## Conclusion
The experiment cleans retail transactions, constructs customer-level features and discovers segments with K-Means. The elbow plot, silhouette scores, Ward dendrogram and PCA plots help assess and interpret the segmentation. Random Forest and SVM are evaluated on held-out customers using accuracy, precision, recall, F1, ROC-AUC and confusion matrices. Feature importance connects the learned classification to purchasing behavior, and cluster-specific campaigns translate profiles into marketing actions.

Select the final model from the executed metrics and business costs, rather than assuming one model always performs better. Segments depend on observation period, preprocessing, feature weighting and cluster count. For deployment, validate segment stability, monitor drift, honor customer communication preferences and measure campaign uplift through controlled experiments.

## Submission checklist
1. Run every cell and confirm that outputs appear without errors.
2. Save the executed notebook. In Colab, use **File → Print** and your browser's **Save as PDF**; interactive 3D content may appear as a static snapshot, so retain the notebook/HTML too.
3. Upload the notebook/code to your own Git repository.
4. Add your repository link below before submission.

**Git repository link:** ADD YOUR REPOSITORY URL HERE
